# SentimentumAI — HuggingFace Inference (Google Colab)

Este notebook corre la inferencia del modelo `cardiffnlp/twitter-roberta-base-hate-latest` sobre el test set completo y exporta los resultados para integrarlos en el notebook principal.

**Pasos:**
1. Subir `twitter_train.csv` cuando se pida
2. Ejecutar todas las celdas en orden
3. Descargar `hf_resultados.csv` al finalizar

> Activar GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`

In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
  print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
  print("Sin GPU")

Device: cuda
GPU: Tesla T4


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
df_twitter = pd.read_csv("/content/Datos/twitter_train.csv")

_, X_test_text, _, y_test = train_test_split(
    df_twitter['tweet'],
    df_twitter['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_twitter['label']
)

tweets_origin = X_test_text.to_list()
y_muestra = y_test.values

print(f"Test set: {len(tweets_origin)} tweets")
print(f"No ODIO: {(y_muestra==0).sum()}  | ODIO: {(y_muestra==1).sum()}")

Test set: 6393 tweets
No ODIO: 5945  | ODIO: 448


In [15]:
from transformers import pipeline as hf_pipeline

print(f"Cargando modelo en {device}...")
hf_model = hf_pipeline(
    task="text-classification",
    model= "cardiffnlp/twitter-roberta-base-hate-latest",
    device= 0 if device == "cuda" else -1,
    truncation= True,
    max_lenght= 120
)
print("Modelo cargado")

BATCH_SIZE= 64
resultados = []

for i in range(0, len(tweets_origin), BATCH_SIZE):
  batch = tweets_origin[i: i+BATCH_SIZE]
  resultados.extend(hf_model(batch))

label_map = {"HATE": 1, "NOT-HATE": 0}
y_pred_hf = np.array([label_map[r["label"]] for r in resultados])
y_prob_hf = np.array([r['score'] if r['label']=='HATE' else 1-r['score'] for r in resultados])

print(f"\n Completado | Predcciones de odio: {(y_pred_hf).sum()}")

Cargando modelo en cuda...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado

 Completado | Predcciones de odio: 295


In [17]:
df_export = pd.DataFrame({
    'tweet':tweets_origin,
    'label_real': y_muestra,
    'label_predict':y_pred_hf,
    'prob_hf': y_prob_hf
})

df_export.to_csv("hf_resultados.csv", index=False)
print(f"hf_resultados.csv exportado correctamente [{len(df_export)} filas]")

hf_resultados.csv exportado correctamente [6393 filas]
